# Library

In [1]:
!pip install -qq transformers underthesea sentencepiece
!pip install -qq torch
!pip install -qq rouge-score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 58.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.6/978.6 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 105.9 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.2 MB/s eta 0:00:0000:010:0

In [2]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from sklearn.cluster import KMeans
from rouge_score import rouge_scorer
import sacrebleu
from underthesea import sent_tokenize, word_tokenize

# Load model

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
# model PhoBert

model = AutoModel.from_pretrained("vinai/phobert-base").to(device)
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
model.eval()

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

# Get sentence embedding

In [5]:
# chuyển từ câu -> vectors embedding 

def get_embeddings(sentences, batch_size=32):
    embeddings  = []
    
    with torch.no_grad():
        # for i in tqdm(range(0, len(sentences), batch_size)):    
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, return_tensors="pt", truncation=True, max_length=256).to(device)
            outputs = model(**inputs)

            #last_hidden_state  trả về emb cho từng token -> lấy trung bình để thành emb của câu
            #nhưng loại bỏ những token padding vào ở attention_mask
            last_hidden = outputs.last_hidden_state   #lấy last_hidden_state 
            mask = inputs["attention_mask"].unsqueeze(-1)   # attention mask
            masked_hidden = last_hidden * mask 
            sum_hidden = masked_hidden.sum(dim=1)
            lengths = mask.sum(dim=1)  
            mean_pooled = sum_hidden / lengths        #mean pooling

            embeddings.append(mean_pooled.cpu().numpy())
            
            # emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()   
            # embeddings.append(emb)

    return np.vstack(embeddings)

# K-mean extractive summary

In [6]:
#Ý tưởng: 1 bài báo chia thành nhiều câu, -> mảng vector emb. Dùng kmean để phân cụm các câu liên quan vào chung nhóm. 
#         Từ mỗi nhóm đó lấy ra 1 câu gần nhất tâm cụm làm câu tóm tắt. Số tâm cụm = số câu tóm tắt


def kmeans_summary(sentences):    #sentences  là mảng các câu sau khi tách câu

    k = int(np.ceil(len(sentences)**0.5))    #số cụm = căn 2 của số câu gốc 
    emb = get_embeddings(sentences)     

    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(emb)

    summary = []
    for cid in range(k):
        idx = np.where(kmeans.labels_ == cid)[0]     
        centroid = kmeans.cluster_centers_[cid]                               #tâm cụm
        best = idx[np.argmin(np.linalg.norm(emb[idx] - centroid, axis=1))]   #chọn ra vector gần tâm cụm nhất theo kc euclid
        summary.append((best, sentences[best]))

    summary = sorted(summary)                                                 
    return " ".join([s for _, s in summary]) 

# DBSCAN extractive summary

In [7]:
#dbscan: phân cụm dựa trên mật độ
# ý tưởng: 1 điểm thuộc 1 cụm nếu nó nawmgf trong vùng có mật độ cao, có ít nhất min_samples điểm khác 

from sklearn.cluster import DBSCAN

def dbscan_summary(sentences, eps=3.7, min_samples=2):    #eps: ngưỡng khoảng cách 2 điểm coi là gần nhau

    emb = get_embeddings(sentences)
    #tính phân bố kcachs
    
    db = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
    labels = db.fit_predict(emb)

    unique_clusters = sorted([c for c in set(labels) if c != -1])  # bỏ cụm nhiễu -1

    #vẫn là chọn câu từ các cụm
    summary = []
    for cid in unique_clusters:
        idx = np.where(labels == cid)[0]
        centroid = emb[idx].mean(axis=0)
        best = idx[np.argmin(np.linalg.norm(emb[idx] - centroid, axis=1))]
        summary.append((best, sentences[best]))

    summary = sorted(summary)
    return " ".join([s for _, s in summary])

# Evaluating (Rouge1, Rouge2, RougeL, Bleu)

In [8]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def evaluate(gold, pred):
    gold = " ".join(word_tokenize(gold))   #tách từ theo tiếng việt rồi ghép lại, vì tiếng việt có từ ghép "học sinh" != "học", "sinh"
    pred = " ".join(word_tokenize(pred))
    
    r = rouge.score(gold, pred)
    bleu = sacrebleu.corpus_bleu([pred], [[gold]]).score

    return {
        "rouge1": round(r["rouge1"].fmeasure, 4),
        "rouge2": round(r["rouge2"].fmeasure, 4),
        "rougeL": round(r["rougeL"].fmeasure, 4),
        
        "bleu": round(bleu / 100, 4)  # sacrebleu ở thang đo 0-100 -> 0-1
    }

# Test with 1 sample 

In [9]:
#sample: text là bản gốc, gold là bản tóm tắt (gold trong Gold Summary Reference)

text = """Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Đại diện Sở Giao thông cho biết mục tiêu là tăng tỷ lệ người dân sử dụng phương tiện công cộng lên 25%.
Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Việc áp dụng vé điện tử được kỳ vọng sẽ giúp giảm thời gian lên xuống xe và hạn chế gian lận.
Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm.
Các chuyên gia cho rằng muốn đạt mục tiêu này, cần đồng thời tăng phí đỗ xe và kiểm soát chặt xe máy cũ.
Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau.
Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.
Tuy nhiên, một số ý kiến lo ngại tiến độ thi công có thể bị chậm do vướng mắc giải phóng mặt bằng."""

gold = """Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm."""

In [10]:
#tách thành từng câu
sentences = sent_tokenize(text)   
print("Số câu:", len(sentences))
for i, s in enumerate(sentences):
    print(f"Câu {i+1}: {s}")

Số câu: 10
Câu 1: Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Câu 2: Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Câu 3: Đại diện Sở Giao thông cho biết mục tiêu là tăng tỷ lệ người dân sử dụng phương tiện công cộng lên 25%.
Câu 4: Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Câu 5: Việc áp dụng vé điện tử được kỳ vọng sẽ giúp giảm thời gian lên xuống xe và hạn chế gian lận.
Câu 6: Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm.
Câu 7: Các chuyên gia cho rằng muốn đạt mục tiêu này, cần đồng thời tăng phí đỗ xe và kiểm soát chặt xe máy cũ.
Câu 8: Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau.
Câu 9: Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.
Câu 10: Tuy 

In [11]:
#lấy embedding cho toàn bộ câu
emb = get_embeddings(sentences)
print("embed shape:", emb.shape)

embed shape: (10, 768)


## KMEAN

In [12]:
# Kmean
#Phân cụm -> tóm tắt
summary = kmeans_summary(sentences)
print("Bản tóm tắt dự đoán:")
print(summary)

Bản tóm tắt dự đoán:
Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới. Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải. Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm. Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.


In [13]:
#đánh giá
scores = evaluate(gold, summary)
print("Đánh giá:")
print(scores)

Đánh giá:
{'rouge1': 0.869, 'rouge2': 0.7986, 'rougeL': 0.7724, 'bleu': 0.7603}


## DBSCAN

In [14]:
#DBSCAN

#Phân bố khoảng cách dữ liệu

D = np.linalg.norm(emb[:, None, :] - emb[None, :, :], axis=2)
print("Min:", D[D>0].min())
print("Max:", D.max())
print("Mean:", D.mean())

Min: 3.347478
Max: 4.834093
Mean: 3.788861


In [15]:
#Phân cụm -> tóm tắt
summary_dbscan = dbscan_summary(sentences, eps=D.mean(), min_samples=2)    #chọn eps còn thùy thuộc vào phôn bố dữ liệu
print("Bản tóm tắt dự đoán:")
print(summary_dbscan)

Bản tóm tắt dự đoán:
Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau. Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.


In [16]:
#đánh giá
scores = evaluate(gold, summary_dbscan)
print("Đánh giá:")
print(scores)

Đánh giá:
{'rouge1': 0.5926, 'rouge2': 0.4206, 'rougeL': 0.5185, 'bleu': 0.1803}


# Evaluate with benchmark

In [17]:
#đang dùng benchmark abstractive do extractive đang hơi sai 
bench_mark_path = "/kaggle/input/dataset/Benchmark/Benchmark/abstractive_summarization_benchmark.csv"
df = pd.read_csv(bench_mark_path)
df = df.dropna(subset=["content", "summary"])
df.reset_index(drop=True, inplace=True)
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  9837 non-null   object
 1   summary  9837 non-null   object
dtypes: object(2)
memory usage: 153.8+ KB
None
                                             content  \
0  Hướng dẫn điều chỉnh giá hợp đồng xây dựng.\nQ...   
1  Vũ Thu Phương xuất hiện ở vị trí vedette. Ngườ...   
2  Nhắc đến cái tên Jenny McCarthy, người ta thườ...   
3  Thực tế, đến nay Bộ Thông tin và Truyền thông ...   
4  Theo trang tin quốc phòng militaryparitet (Nga...   

                                             summary  
0  Bài báo nhằm hướng dẫn cách điều chỉnh giá hợp...  
1  Vũ Thu Phương xuất hiện với vai trò vedette tr...  
2  Jenny McCarthy được đánh giá là người phụ nữ đ...  
3  Bộ Thông tin và Truyền thông chưa cung cấp hướ...  
4  Ngày 6/12, máy bay vận tải An-124-100M Ruslan ...  


## Kmean Results

In [18]:
# Kmean
all_scores = []

for idx, row in df.iterrows():
    text = row["content"]
    gold = row["summary"]

    sentences = sent_tokenize(text)

    pred = kmeans_summary(sentences)

    scores = evaluate(gold, pred)
    all_scores.append(scores)

results_df = pd.DataFrame(all_scores)
print(results_df.mean())

rouge1    0.675127
rouge2    0.421692
rougeL    0.435602
bleu      0.228550
dtype: float64


## DBSCAN Results

In [19]:
#DBSCAN
all_scores_dbscan = []

for idx, row in df.iterrows():
    text = row["content"]
    gold = row["summary"]

    sentences = sent_tokenize(text)
    
    if len(sentences) <= 1:
        pred = sentences[0] if sentences else ""
        scores = evaluate(gold, pred)
        all_scores_dbscan.append(scores)
        continue
        
    emb = get_embeddings(sentences)
    D = np.linalg.norm(emb[:, None, :] - emb[None, :, :], axis=2)
    
    D_nonzero = D[D > 0]
    if len(D_nonzero) == 0:
        eps = 3.7
    else:
        eps = D_nonzero.mean()
        
    pred = dbscan_summary(sentences, eps=eps, min_samples=2)

    scores = evaluate(gold, pred)
    all_scores_dbscan.append(scores)

results_dbscan = pd.DataFrame(all_scores_dbscan)
print(results_dbscan.mean())

rouge1    0.496993
rouge2    0.302821
rougeL    0.341033
bleu      0.096080
dtype: float64


# Documents

In [20]:
# https://huggingface.co/transformers/v4.3.3/model_doc/phobert.html#overview
# https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans
# https://phamdinhkhanh.github.io/deepai-book/ch_ml/DBSCAN.html
# https://phamdinhkhanh.github.io/2020/06/04/PhoBERT_Fairseq.html#3-load-model-bert